# Notebook 7: YOLOv8 Variant Comparison (n / s / m) — Two-Stage Transfer Learning

Trains all three variants with **progressive transfer learning**:

  Stage 1:  COCO-pretrained  →  fine-tune on the public Indonesian plate dataset (958 imgs)
  Stage 2:  Stage-1 weights  →  fine-tune on the MIPA dataset (204 imgs)

This matches the transfer-learning method described in the proposal and gives the
detector ~1000 plate examples to learn "plate-ness" before specializing on MIPA.
All three variants use the identical two-stage recipe for a fair comparison.

## 1. Setup & mount Drive

In [ ]:
!pip install -q ultralytics
from google.colab import drive
drive.mount('/content/drive')

import torch
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
else:
    print('!! No GPU — change runtime to T4 GPU before continuing.')

DRIVE_ROOT  = '/content/drive/MyDrive/ANPR_MIPA_UGM'
WEIGHTS_DIR = f'{DRIVE_ROOT}/weights'

import os
os.makedirs(WEIGHTS_DIR, exist_ok=True)
print('Drive mounted.')

## 2. Download & unzip BOTH datasets

Two-stage training needs both zips in `ANPR_MIPA_UGM/datasets/` on Drive:
- `public_v1_dataset.zip` — public Indonesian plates (958 train) → Stage 1
- `mipa_v1_dataset.zip`   — MIPA photos (204 train) → Stage 2

Upload both before running this cell.

In [ ]:
import zipfile, shutil, os
from pathlib import Path
import yaml

def fetch_and_unzip(zip_name, dest):
    zip_on_drive = f'{DRIVE_ROOT}/datasets/{zip_name}'
    assert os.path.exists(zip_on_drive), f'Zip not found on Drive: {zip_on_drive}'
    local_zip = f'/content/{zip_name}'
    print(f'Copying {zip_name} -> local disk ...')
    shutil.copy2(zip_on_drive, local_zip)
    print(f'Unzipping {zip_name} ...')
    with zipfile.ZipFile(local_zip, 'r') as zf:
        zf.extractall(dest)
    cands = list(Path(dest).rglob('data.yaml'))
    assert cands, f'No data.yaml found inside {zip_name}'
    return str(cands[0])

PUBLIC_YAML = fetch_and_unzip('public_v1_dataset.zip', '/content/public')
MIPA_YAML   = fetch_and_unzip('mipa_v1_dataset.zip',   '/content/mipa')
DATA_YAML   = MIPA_YAML   # all evaluation is on the MIPA test set

print(f'\nPUBLIC_YAML: {PUBLIC_YAML}')
print(f'MIPA_YAML  : {MIPA_YAML}')

## 3. Fix data.yaml paths (both datasets)

The zips were created on a Mac with absolute paths. Rewrite each `path:` to point
at Colab's local disk so YOLO can find the images.

In [ ]:
import yaml

def fix_path(yaml_path):
    with open(yaml_path) as f:
        cfg = yaml.safe_load(f)
    cfg['path'] = str(Path(yaml_path).parent)
    with open(yaml_path, 'w') as f:
        yaml.dump(cfg, f)
    return cfg

for yp in [PUBLIC_YAML, MIPA_YAML]:
    cfg = fix_path(yp)
    print(f'{yp}\n   path={cfg["path"]}  names={cfg["names"]}  nc={cfg.get("nc")}')

## 4. Two-stage training hyperparameters (tuned for validation mAP)

Stage 1 (public, 958 imgs): learn generic plate features from COCO.
Stage 2 (MIPA, 204 imgs): specialize. The Stage-2 recipe is tuned to lift
validation mAP toward the ≥85% target:

- **close_mosaic=15** — mosaic augmentation distorts images; disabling it for the
  last 15 epochs lets the model fine-tune on clean, un-mosaicked images. This is
  the standard trick for squeezing out the final 1-3 mAP points on val/test.
- **epochs 50 → 100, patience 25** — only 204 training images, so the model was
  slightly underfit; more epochs + patience let it converge.
- **lr0 0.0005 → 0.0003 (gentler)** — two-stage fine-tuning was *eroding* val mAP
  for n and m (0.855→0.811, 0.856→0.819); a softer LR preserves the useful
  Stage-1 features instead of overwriting them.

In [ ]:
COMMON = dict(
    imgsz     = 640,
    optimizer = 'AdamW',
    lrf       = 0.01,
    cos_lr    = True,
    save      = True,
    device    = 0 if torch.cuda.is_available() else 'cpu',
    workers   = 2,
    cache     = True,
    hsv_h=0.01, hsv_s=0.5, hsv_v=0.3,
    degrees=5.0, translate=0.05, scale=0.2,
    fliplr=0.0, mixup=0.0,
)

# Stage 1: COCO -> public Indonesian plates (learn generic plate features)
STAGE1_ARGS = dict(
    epochs=50, batch=8, lr0=0.001, patience=15,
    mosaic=0.5, close_mosaic=10, plots=False, **COMMON,
)

# Stage 2: public -> MIPA (specialize). Tuned for validation mAP:
#   close_mosaic=15  -> clean fine-tuning in the final epochs (+val mAP)
#   epochs=100, patience=25 -> more convergence on the small 204-img set
#   lr0=0.0003 (gentler) -> preserve Stage-1 features rather than erase them
STAGE2_ARGS = dict(
    epochs=100, batch=4, lr0=0.0003, patience=25,
    mosaic=0.5, close_mosaic=15, plots=True, **COMMON,
)

print('Tuned two-stage hyperparameters set.')
print('Stage 1:', STAGE1_ARGS)
print('Stage 2:', STAGE2_ARGS)

## 5. Two-stage training helper + YOLOv8n

`two_stage_train()` runs Stage 1 (public) then Stage 2 (MIPA) and saves the final
MIPA-specialized weights to Drive. Each stage's runs are kept separately under
`/content/runs/` so you can inspect both.

In [ ]:
from ultralytics import YOLO
import shutil

def two_stage_train(variant):
    print(f'\n===== {variant}: STAGE 1  (COCO -> public, 958 imgs) =====')
    m = YOLO(f'{variant}.pt')                      # COCO-pretrained
    m.train(data=PUBLIC_YAML, project='/content/runs',
            name=f'{variant}_stage1_public', **STAGE1_ARGS)
    stage1_best = f'/content/runs/{variant}_stage1_public/weights/best.pt'

    print(f'\n===== {variant}: STAGE 2  (public -> MIPA, 204 imgs) =====')
    m = YOLO(stage1_best)                          # start from Stage-1 weights
    m.train(data=MIPA_YAML, project='/content/runs',
            name=f'{variant}_stage2_mipa', **STAGE2_ARGS)
    stage2_best = f'/content/runs/{variant}_stage2_mipa/weights/best.pt'

    shutil.copy2(stage2_best, f'{WEIGHTS_DIR}/best_{variant}.pt')
    print(f'Saved -> {WEIGHTS_DIR}/best_{variant}.pt')
    return stage2_best

best_n = two_stage_train('yolov8n')

## 6. Two-stage training — YOLOv8s

In [ ]:
best_s = two_stage_train('yolov8s')

## 7. Two-stage training — YOLOv8m

In [ ]:
best_m = two_stage_train('yolov8m')

## 8. Evaluate all three variants on the MIPA test set

In [ ]:
import time
from pathlib import Path

VARIANTS = [
    ('YOLOv8n', best_n),
    ('YOLOv8s', best_s),
    ('YOLOv8m', best_m),
]

with open(DATA_YAML) as f:
    cfg = yaml.safe_load(f)
dataset_root = Path(cfg['path'])
test_images  = sorted(p for p in (dataset_root / 'test' / 'images').glob('*')
                      if p.suffix.lower() in ('.jpg', '.jpeg', '.png'))
print(f'Test images for benchmark: {len(test_images)}')

results_table = []

for name, weights in VARIANTS:
    print(f'\n--- Evaluating {name} ---')
    m = YOLO(weights)

    metrics = m.val(data=DATA_YAML, split='test', imgsz=640,
                    device='cpu', verbose=False)
    map50   = metrics.box.map50
    map5095 = metrics.box.map
    prec    = metrics.box.mp
    rec     = metrics.box.mr

    # warmup then time
    m.predict(str(test_images[0]), imgsz=640, device='cpu', verbose=False)
    t0 = time.perf_counter()
    for img_path in test_images:
        m.predict(str(img_path), imgsz=640, device='cpu', verbose=False)
    avg_ms = (time.perf_counter() - t0) / len(test_images) * 1000

    results_table.append({
        'variant': name, 'map50': map50, 'map5095': map5095,
        'precision': prec, 'recall': rec, 'cpu_ms': avg_ms,
    })
    print(f'  mAP50={map50:.4f}  mAP50-95={map5095:.4f}  P={prec:.4f}  R={rec:.4f}  CPU={avg_ms:.0f}ms/img')

print('\nAll variants evaluated.')

## 9. Print comparison table

In [ ]:
print('\n' + '='*72)
print('  YOLOV8 VARIANT COMPARISON — MIPA TEST SET')
print('='*72)
header = f"{'Variant':<12} {'mAP@0.5':>9} {'mAP@0.5:0.95':>14} {'Precision':>10} {'Recall':>8} {'CPU ms/img':>11}"
print('  ' + header)
print('  ' + '-'*68)
for r in results_table:
    row = (f"{r['variant']:<12} {r['map50']:>9.4f} {r['map5095']:>14.4f}"
           f" {r['precision']:>10.4f} {r['recall']:>8.4f} {r['cpu_ms']:>10.0f}")
    print('  ' + row)
print('='*72)

DET_TIME_BUDGET_MS = 1500
candidates = [r for r in results_table if r['cpu_ms'] < DET_TIME_BUDGET_MS]
if candidates:
    winner = max(candidates, key=lambda r: r['map50'])
    print(f"\n  Recommended: {winner['variant']} (best mAP50 under {DET_TIME_BUDGET_MS}ms)")
    print(f"  mAP50={winner['map50']:.4f}, CPU={winner['cpu_ms']:.0f}ms/img")
else:
    best = max(results_table, key=lambda r: r['map50'])
    print(f"\n  All variants exceed {DET_TIME_BUDGET_MS}ms; pick by accuracy: {best['variant']}")

## 10. Save results CSV to Drive

In [ ]:
import csv

csv_path = f'{DRIVE_ROOT}/variant_comparison.csv'
with open(csv_path, 'w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=['variant','map50','map5095','precision','recall','cpu_ms'])
    writer.writeheader()
    writer.writerows(results_table)

print(f'Results saved -> {csv_path}')
print('\n--- Download from Drive into the repo ---')
print(f'  {WEIGHTS_DIR}/best_yolov8n.pt  -> models/')
print(f'  {WEIGHTS_DIR}/best_yolov8s.pt  -> models/')
print(f'  {WEIGHTS_DIR}/best_yolov8m.pt  -> models/')
print(f'  {csv_path}                      -> data/')
print('\nNOTE: all three are two-stage (COCO -> public -> MIPA).')
print('Final authoritative eval is run locally on the curated 30-img MIPA test set.')

## 11. Visual check — predictions on test images (optional)

In [ ]:
import cv2
import matplotlib.pyplot as plt

preview_imgs = test_images[:4]
fig, axes = plt.subplots(3, 4, figsize=(20, 12))

for row_idx, (name, weights) in enumerate(VARIANTS):
    m = YOLO(weights)
    for col_idx, img_path in enumerate(preview_imgs):
        res = m.predict(str(img_path), conf=0.25, imgsz=640,
                        device='cpu', verbose=False)[0]
        img = cv2.cvtColor(res.plot(), cv2.COLOR_BGR2RGB)
        axes[row_idx][col_idx].imshow(img)
        axes[row_idx][col_idx].axis('off')
        if col_idx == 0:
            axes[row_idx][col_idx].set_ylabel(name, fontsize=12, fontweight='bold')
        if row_idx == 0:
            axes[row_idx][col_idx].set_title(img_path.name[:20], fontsize=8)

plt.suptitle('Variant comparison — rows: n / s / m, cols: test images', fontsize=13)
plt.tight_layout()
plt.savefig(f'{DRIVE_ROOT}/variant_comparison_preview.png', dpi=100)
plt.show()
print('Preview saved to Drive.')